In [ ]:
import ee
import os
import calendar
import requests
import numpy as np
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable

# 1. Inicialización
# ee.Authenticate() # Solo si no lo has hecho antes
ee.Initialize()

# 2. Crear el directorio de resultados
carpeta_salida = 'Resultados_Calor_Estatal'
os.makedirs(carpeta_salida, exist_ok=True)

# 3. Definir Parámetros y Área
paises = ['France', 'Spain', 'Portugal', 'Andorra', 'Monaco']
aoi = ee.FeatureCollection("FAO/GAUL/2015/level0") \
        .filter(ee.Filter.inList('ADM0_NAME', paises)) \
        .union(100).geometry()

start_year = 2019
end_year = 2023

# 4. Funciones de Procesamiento GEE
def obtener_maximas_mensuales():
    meses_totales = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start_date = ee.Date.fromYMD(year, month, 1)
            end_date = start_date.advance(1, 'month')
            
            dataset = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
                .filterBounds(aoi) \
                .filterDate(start_date, end_date) \
                .select('temperature_2m')
                
            max_temp = dataset.max().subtract(273.15) \
                .set('year', year) \
                .set('month', month) \
                .set('system:time_start', start_date.millis()) \
                .clip(aoi)
            meses_totales.append(max_temp)
    return ee.ImageCollection.fromImages(meses_totales)

print("Calculando datos base en Earth Engine...")
coleccion_mensual = obtener_maximas_mensuales()

# Asumimos normalidad temporal píxel por píxel
media_temporal = coleccion_mensual.mean()
std_temporal = coleccion_mensual.reduce(ee.Reducer.stdDev())

# Paleta Turbo ajustada en GEE (simulando desde azul glacial hasta rojo intenso)
# En Earth Engine, no existe 'turbo' nativo por nombre, así que pasamos los colores hexadecimales aproximados
turbo_palette = ['#30123b', '#4662d7', '#36aaf9', '#1ae4b6', '#72fe5e', 
                 '#c8ef34', '#faba39', '#f66b19', '#cb2a04', '#7a0403']

# 5. Bucle de Descarga y Renderizado con Matplotlib
lista_imagenes = coleccion_mensual.toList(coleccion_mensual.size())
num_imagenes = lista_imagenes.size().getInfo()

print(f"Generando y descargando {num_imagenes} imágenes. Esto puede tomar unos minutos...")

for i in range(num_imagenes):
    # Extraer imagen y metadatos
    img = ee.Image(lista_imagenes.get(i))
    year = img.get('year').getInfo()
    month = img.get('month').getInfo()
    nombre_mes = calendar.month_name[month].capitalize()
    
    # Calcular Z-Score
    z_score = img.subtract(media_temporal).divide(std_temporal)
    
    # Visualizar en GEE (-2 a +2 SD) y obtener URL de la imagen
    vis_img = z_score.visualize(min=-2, max=2, palette=turbo_palette)
    url = vis_img.getThumbURL({
        'region': aoi,
        'dimensions': 1000, # Resolución en píxeles de la imagen descargada
        'format': 'png'
    })
    
    # Descargar la imagen a la memoria
    respuesta = requests.get(url)
    imagen_pil = Image.open(BytesIO(respuesta.content))
    
    # --- MAQUETACIÓN CON MATPLOTLIB ---
    fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
    
    # Quitar ejes del mapa
    ax.imshow(imagen_pil)
    ax.axis('off')
    
    # Título programático
    titulo = f"Temperaturas Normalizadas Periodo {start_year} a {end_year}\n{nombre_mes} {year}"
    ax.set_title(titulo, fontsize=14, fontweight='bold', pad=15)
    
    # Marca de agua "Anaplian" en la esquina inferior derecha
    # Usamos coordenadas relativas de los ejes (1.0 = borde derecho, 0.0 = borde inferior)
    ax.text(0.98, 0.02, 'Anaplian', transform=ax.transAxes, 
            fontsize=12, color='white', fontweight='bold', 
            ha='right', va='bottom', bbox=dict(facecolor='black', alpha=0.5, edgecolor='none'))
    
    # Añadir la barra de colores (Z-Score)
    # Recreamos el espectro 'turbo' de matplotlib para la leyenda
    cmap = plt.get_cmap('turbo')
    norm = mcolors.Normalize(vmin=-2, vmax=2)
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    
    # Ajustar la posición de la barra
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.04)
    cbar.set_label('Desviaciones Estándar (Z-Score)', fontsize=11, fontweight='bold')
    
    # Guardar imagen localmente
    nombre_archivo = f"{year}_{month:02d}_Anomalia.png"
    ruta_guardado = os.path.join(carpeta_salida, nombre_archivo)
    
    plt.tight_layout()
    plt.savefig(ruta_guardado, bbox_inches='tight', facecolor='white')
    plt.close(fig) # Cerramos la figura para liberar memoria
    
    print(f"Guardado: {nombre_archivo}")

print(f"\n¡Proceso completado! Tus imágenes están en la carpeta '{carpeta_salida}'.")